# When worlds collide (again): SQP optimization under uncertainty

In the [previous SQP notebook](freyberg_sqp_1.ipynb) we used `PESTPP-SQP` to solve a _risk-neutral_ management optimization problem: maximize future groundwater extraction subject to ecological and minimum-supply constraints, using a single set of "best-estimate" (calibrated) parameters.

But - just as in the [PESTPP-OPT under uncertainty notebook](../part2_08_opt/freyberg_opt_2.ipynb) - trusting a single set of parameters ignores everything those hard-won history-matching notebooks taught us: model predictions are _uncertain_. If parameter uncertainty means our "optimal" pumping actually pushes the stream over the ecological limit some of the time, then the risk-neutral answer is dangerously optimistic.

So in this notebook we carry parameter uncertainty into the optimization. `PESTPP-SQP` supports _chance constraints_: instead of requiring the best-estimate constraint value to be feasible, we require it to be feasible _with some level of confidence_, given the uncertainty in the model. We supply that uncertainty as a "stack" - an ensemble of parameter realizations.

Once again, we do not need to build any of this from scratch. The [PESTPP-OPT under uncertainty notebook](../part2_08_opt/freyberg_opt_2.ipynb) already did the work: it took the posterior parameter ensemble from `PESTPP-IES`, turned it into a parameter stack, and configured the chance-constraint machinery (the stack file and a risk stance) in its template folder. We simply reuse that setup and swap the solver from `PESTPP-OPT` to `PESTPP-SQP`.

We will then watch the "cost of uncertainty" appear: to be confident the ecological constraint holds, we have to leave more water in the ground, so the optimal extraction drops relative to the risk-neutral case.

As before, this is ensemble-based: __you choose__ the size of the decision-variable ensemble (`sqp_num_reals`, for the StoSAG gradients), while the parameter stack (from PESTPP-IES) carries the constraint uncertainty.

### Admin

Same old admin. Load dependencies.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
font = {'size'   : 10}
import matplotlib
matplotlib.rc('font', **font)
import matplotlib.pyplot as plt;
import shutil
import psutil

import sys
import pyemu
import flopy
assert "dependencies" in flopy.__file__
assert "dependencies" in pyemu.__file__
sys.path.insert(0,"..")
import herebedragons as hbd

### Reuse the PESTPP-OPT chance-constrained setup

The ["freyberg opt 2"](../part2_08_opt/freyberg_opt_2.ipynb) notebook prepared a chance-constrained PEST interface with:

- the same decision variables, objective, and constraints as the risk-neutral problem,
- a _parameter stack_ (`par_stack.csv`) built from the `PESTPP-IES` posterior ensemble, and
- a _risk_ stance (`opt_risk = 0.95`) - i.e. require 95% confidence that the constraints hold.

It left all of that in its `freyberg6_template_chance` folder, so we copy it across. Run the next cell - it will complain if you have not run the PESTPP-OPT under uncertainty notebook yet.

In [ ]:
# specify the temporary working folder
t_d = os.path.join('freyberg6_template_chance')
if os.path.exists(t_d):
    shutil.rmtree(t_d)

# the pestpp-opt chance template, which already has the decision variables, objective,
# constraints, the ies-based parameter stack, and the risk stance set up
org_t_d = os.path.join("..","part2_08_opt","freyberg6_template_chance")
assert os.path.exists(org_t_d), "you need to run the '/part2_08_opt/freyberg_opt_2.ipynb' notebook first"

shutil.copytree(org_t_d,t_d)
pst_path = os.path.join(t_d, 'pest.pst')

In [ ]:
pst = pyemu.Pst(pst_path)

Take a quick look at the chance-constraint machinery the PESTPP-OPT notebook left for us:

In [ ]:
print("parameter stack file:", pst.pestpp_options["opt_par_stack"])
print("stack file exists:    ", os.path.exists(os.path.join(t_d,pst.pestpp_options["opt_par_stack"])))
print("risk stance:          ", pst.pestpp_options["opt_risk"])
stack = pd.read_csv(os.path.join(t_d,pst.pestpp_options["opt_par_stack"]),index_col=0)
print("stack size (realizations x parameters):", stack.shape)

### How chance constraints work here

The _stack_ is an ensemble of parameter realizations (the `PESTPP-IES` posterior). `PESTPP-SQP` runs the model for each stack realization and uses the resulting _spread_ in the constraint outputs to empirically estimate constraint uncertainty. It then _shifts_ each constraint by an amount that depends on the chosen `risk` value, so that the optimization respects the constraint with the desired confidence.

The `risk` value sets how conservative we are, on a scale from 0 to 1:

- `risk = 0.5` is _risk-neutral_ - the best-estimate (median) constraint value, like the previous notebook.
- `risk > 0.5` is _risk-averse_ - require the constraint to hold even for the less-favourable tail of the uncertainty distribution. `risk = 0.95` means "I want to be 95% confident the constraint holds."
- `risk < 0.5` is _risk-tolerant_ - a gambler's stance.

The PESTPP-OPT notebook already set `opt_risk = 0.95`. Feel free to change it and re-run to explore how the optimal solution shifts with your risk appetite.

In [ ]:
pst.pestpp_options["opt_risk"] = 0.95  # 95% confidence the constraints hold; change to explore your risk stance

### The SQP-specific bits

As in the risk-neutral notebook, we (1) start the decision variables at a feasible interior point (multiplier 1.0), and (2) turn on ensemble gradients by choosing `sqp_num_reals`. Note that this is a _separate_ ensemble from the parameter stack: `sqp_num_reals` sets the size of the decision-variable ensemble used to estimate the StoSAG gradients, while the stack carries the parameter uncertainty for the chance constraints.

In [ ]:
par = pst.parameter_data
dv_names = par.loc[par.pargp=="decvars","parnme"]
par.loc[dv_names,"parval1"] = 1.0  # feasible starting point (current pumping)

In [ ]:
num_reals = 30 # the decision-variable ensemble size for StoSAG gradients - choose to suit your resources!
pst.pestpp_options["sqp_num_reals"] = num_reals

## An aside on "coupling"

We are jamming two very different decision-support concepts together into a single algorithm, and there are knock-on effects. The biggest is _coupling_ between the decision variables and the constraint uncertainty.

During _history matching_ we learned the relation between model parameters and model outputs (the observations). During _optimization_ we exploit the relation between decision variables and model outputs (the constraints). Chance-constrained optimization has to consider both at once: the uncertainty in the constraints (from parameter uncertainty) can itself depend on where the decision variables are. `PESTPP-SQP` handles this by re-evaluating the stack as the optimization proceeds; how often it does so is a trade-off between fidelity and cost (see the `opt_recalc_chance_every` option). For this tutorial we keep it simple, but be aware that for strongly coupled problems this interaction matters.

## Run PESTPP-SQP

Let's first check the setup is valid with a `noptmax=0` run, then re-write with our optimization iteration count.

In [ ]:
pst.control_data.noptmax = 0
pst.write(pst_path,version=2)
pyemu.os_utils.run("pestpp-sqp pest.pst",cwd=t_d)

In [ ]:
pst.control_data.noptmax = 3
pst.write(pst_path,version=2)

# Attention!

You must specify a number of workers which is adequate for ***your*** machine! Make sure to assign an appropriate value for the following `num_workers` variable:

In [ ]:
num_workers = 10 # update according to your available resources!

In [ ]:
m_d = os.path.join('master_sqp_2')

Deploy the agents and manager and start the run. This one does more work than the risk-neutral case, because in addition to the decision-variable ensemble it also runs the parameter stack to estimate constraint uncertainty.

In [ ]:
pyemu.os_utils.start_workers(t_d,"pestpp-sqp","pest.pst",num_workers=num_workers,worker_root=".",
                           master_dir=m_d)

### Processing PESTPP-SQP

As in the previous notebook, we summarize each iteration's ensemble by its mean. The helper below grabs the per-iteration decision-variable and model-output ensembles.

In [ ]:
def iter_ensemble_files(m_d,suffix):
    """list the per-iteration ensemble files (pest.<N><suffix>) sorted by iteration number"""
    found = []
    for f in os.listdir(m_d):
        if f.startswith("pest.") and f.endswith(suffix):
            mid = f[len("pest."):-len(suffix)]
            if mid.isdigit():
                found.append((int(mid),f))
    return [f for _,f in sorted(found)]

par_files = iter_ensemble_files(m_d,".par.csv")
obs_files = iter_ensemble_files(m_d,".obs.csv")

#### Objective function history (risk-averse)

The objective is total future extraction (the sum of the pumping multipliers). Watch it climb as `PESTPP-SQP` marches the ensemble uphill - but under the 95% chance constraints this time.

In [ ]:
obj_mean,obj_lo,obj_hi = [],[],[]
for f in par_files:
    pe = pd.read_csv(os.path.join(m_d,f),index_col=0)
    obj_real = pe.loc[:,dv_names.values].sum(axis=1)
    obj_mean.append(obj_real.mean()); obj_lo.append(obj_real.min()); obj_hi.append(obj_real.max())

fig,ax = plt.subplots(1,1,figsize=(7,4))
its = np.arange(len(obj_mean))
ax.fill_between(its,obj_lo,obj_hi,alpha=0.2,label="ensemble spread")
ax.plot(its,obj_mean,marker='o',label="ensemble mean")
ax.set_xlabel("SQP iteration")
ax.set_ylabel("objective function\n(total future extraction multiplier)")
ax.set_title("PESTPP-SQP objective history (risk-averse)")
ax.legend(); ax.grid()
plt.tight_layout()
plt.show()

#### The risk-averse optimal solution

The optimal decision variables are the ensemble mean of the final iteration; the constraint values are the mean of the final model-output ensemble.

In [ ]:
obs = pst.observation_data
swgw_constraint_names = obs.loc[obs.obgnme=="less_than_swgw","obsnme"].tolist()
wel_constraint_names = obs.loc[obs.obgnme=="less_than_wel","obsnme"].tolist()
swgw_rhs = obs.loc[swgw_constraint_names,"obsval"].max()
wel_rhs = obs.loc[wel_constraint_names,"obsval"].max()

pe_final = pd.read_csv(os.path.join(m_d,par_files[-1]),index_col=0)
opt = pe_final.loc[:,dv_names.values].mean()
oe_final = pd.read_csv(os.path.join(m_d,obs_files[-1]),index_col=0)

# organise the optimal (mean) decision variables by well and stress period
wpar = par.loc[dv_names,:].copy()
wpar["inst"] = wpar.inst.astype(int)
wpar["kij"] = wpar.apply(lambda x: (x.idx0,x.idx1,x.idx2),axis=1)
wpar["optimal"] = opt.loc[wpar.parnme].values
inst_vals = sorted(wpar.inst.unique())
vals = {}
for inst in inst_vals:
    ipar = wpar.loc[wpar.inst==inst,:].copy()
    ipar.sort_values(by="kij",inplace=True)
    ipar.index = ipar.kij
    vals[inst] = ipar.optimal

In [ ]:
swgw_opt = oe_final[swgw_constraint_names].mean()
wel_opt = oe_final[wel_constraint_names].mean()

fig,axes = plt.subplots(2,1,figsize=(12,6))
colors = ["r","g","b","c","m","y","0.5"]
df = pd.DataFrame(vals).T
df.plot(ax=axes[0],kind="bar",color=colors)
axes[0].set_ylim(0,6.5)
axes[0].set_title("risk-averse optimal (mean) pumping multiplier per well and future stress period")
axes[0].set_ylabel("pumping multiplier")
if axes[0].get_legend() is not None:
    axes[0].get_legend().remove()

axes[1].plot(np.arange(len(wel_constraint_names)),wel_opt.values,"b",lw=1.5,label="sim water use")
axes[1].plot(axes[1].get_xlim(),[wel_rhs,wel_rhs],"b--",lw=2.5,label="water-use constraint")
axt = plt.twinx(axes[1])
axt.plot(np.arange(len(swgw_constraint_names)),swgw_opt.values,"m",lw=1.5,label="sim sw-gw")
axt.plot(axes[1].get_xlim(),[swgw_rhs,swgw_rhs],"m--",lw=2.5,label="sw-gw constraint")
axes[1].set_xlabel("future stress period")
axes[1].set_ylabel("water use ($L^3/T$)",color="b")
axt.set_ylabel("sw-gw exchange ($L^3/T$)",color="m")
axes[1].set_title("constraints at the risk-averse optimal solution")
plt.tight_layout()
plt.show()

### The cost of uncertainty

Compare this risk-averse solution to the risk-neutral one from the [previous notebook](freyberg_sqp_1.ipynb). Because we now insist on being 95% confident the ecological constraint holds - despite parameter uncertainty - `PESTPP-SQP` has to keep the simulated sw-gw exchange comfortably on the safe side of zero, leaving a buffer for the uncertainty. That buffer costs us water: the risk-averse optimal extraction is _lower_ than the risk-neutral optimum.

That difference - the water we forgo in exchange for confidence - is the __cost of uncertainty__. It is exactly the kind of quantity a decision-maker needs: not just "how much can we pump?", but "how much can we pump _and still be confident we won't harm the stream_?"

Try re-running with different `opt_risk` values (e.g. 0.5, 0.8, 0.95) to trace out how the optimal extraction trades off against your confidence in the outcome.